In [ ]:
import os
at_colab = "COLAB_GPU" in os.environ

if at_colab:
    %pip install pygadm
    %pip install owslib
    %pip install distinctipy
    %pip install folium
    %pip install mapclassify
    %pip install selenium webdriver-manager

In [ ]:
# Standard library imports
import xml.etree.ElementTree as ET
import requests
import io
from pathlib import Path

# Third-party library imports
import geopandas as gpd
import pygadm
import distinctipy
import matplotlib.colors as mcolors
from owslib.wfs import WebFeatureService
from PIL import Image

In [ ]:
!wget -O jg_folium.py "https://raw.githubusercontent.com/gromicho/tools/refs/heads/main/jg_folium.py"


In [ ]:
import jg_folium as jg

In [ ]:
def folium_to_png(folium_map, file_name, rendering_seconds=5, dpi=300):
    """
    Convert a Folium map to a PNG image.

    Parameters:
        folium_map (folium.Map): The Folium map object.
        file_name (str): Output filename (with or without .png extension).
        rendering_seconds (int, optional): Time to wait for rendering. Default is 5.
        dpi (int, optional): DPI for the saved image. Default is 300.

    Returns:
        str: The path to the saved image.
    """

    file_path = Path(file_name).with_suffix(".png")

    # Generate PNG image
    img_data = folium_map._to_png(rendering_seconds)
    img = Image.open(io.BytesIO(img_data))

    # Save image
    img.save(file_path, dpi=(dpi, dpi))

    return str(file_path)

# First attempt: from GADM

In [ ]:
country_name = "Netherlands"
ad_level = 2

# Download the shapefile data
gdf_gadm = pygadm.Items(name=country_name, content_level=ad_level)

In [ ]:
# Reproject to EPSG:28992
if not gdf_gadm.crs:
    gdf_gadm.set_crs(epsg=4326, inplace=True, allow_override=True)  # Define original CRS
gdf_gadm = gdf_gadm.to_crs(epsg=28992)  # Convert to Dutch RD New (EPSG:28992)

In [ ]:
gdf_merged = gdf_gadm.dissolve(by="NAME_2").reset_index()
gdf_merged = gdf_merged[gdf_merged.ENGTYPE_2 == 'Municipality']

In [ ]:
city_names_gadm = sorted(set(gdf_merged["NAME_2"]))
colors = distinctipy.get_colors(len(city_names_gadm))

In [ ]:
color_dict = {
    name: mcolors.to_hex(colors[i]) 
    for i, name in enumerate(city_names_gadm)
}

In [ ]:
gdf_merged["color"] = gdf_merged["NAME_2"].map(color_dict)

map_all_cities_gadm = gdf_merged.rename(
    columns={'NAME_1':'Province','NAME_2':'Municipality'}
).explore(
    color=gdf_merged["color"],
    legend=False,
    tiles="OpenStreetMap",
    tooltip=['Municipality','Province'],
    zoom_on_bounds=True
)

In [ ]:
map_all_cities_gadm

In [ ]:
folium_to_png( map_all_cities_gadm, 'map_all_cities_gadm' )

In [ ]:
gdf_capelle = gdf_merged[gdf_merged.NAME_2.str.contains('Capelle')]
map_capelle_gadm = gdf_capelle.explore(
    tooltip=False,
    style_kwds={'color': 'orange', 'weight': 5, 'fill': False}
)

In [ ]:
folium_to_png( map_capelle_gadm, 'map_capelle_gadm' )

In [ ]:
# Define the WFS GetCapabilities URL
wfs_url = "https://service.pdok.nl/cbs/gebiedsindelingen/2025/wfs/v1_0"
get_capabilities_url = f"{wfs_url}?service=WFS&version=2.0.0&request=GetCapabilities"

# Download the GetCapabilities XML document
response = requests.get(get_capabilities_url)

if response.status_code == 200:
    # Parse the XML content
    root = ET.fromstring(response.content)

    # Extract all FeatureType names
    namespaces = {'wfs': 'http://www.opengis.net/wfs/2.0'}
    feature_types = root.findall(".//wfs:FeatureType", namespaces)

    # Print the available FeatureType names
    print("🔹 Available Feature Types:")
    for feature in feature_types:
        name = feature.find("wfs:Name", namespaces)
        if name is not None:
            print(f"  - {name.text}")
else:
    print(f"❌ Failed to download GetCapabilities XML. HTTP Status Code: {response.status_code}")


In [ ]:
# Extract and filter FeatureType names using list comprehension
filtered_names = [
    feature.find("wfs:Name", namespaces).text
    for feature in feature_types
    if feature.find("wfs:Name", namespaces) is not None
]

possible_shapes = [name for name in filtered_names if 'gemeente' in name.lower()]

In [ ]:
def get_pdok( 
    layer_name,
    wfs_url = "https://service.pdok.nl/cbs/gebiedsindelingen/2025/wfs/v1_0" 
):
    # Connect to the WFS service
    wfs = WebFeatureService(url=wfs_url, version="2.0.0")

    # Fetch the data as GeoJSON
    params = {
        "service": "WFS",
        "version": "2.0.0",
        "request": "GetFeature",
        "typeName": layer_name,
        "outputFormat": "application/json"
    }

    # Make the request
    response = requests.get(wfs_url, params=params)

    # Check if request was successful
    if response.status_code == 200:
        # Load response into a GeoDataFrame
        return gpd.read_file(io.StringIO(response.text))
    else:
        print(f"❌ WFS request failed with status code {response.status_code}")
        print(f"Error message: {response.text}")
    return None

In [ ]:
cbs = { name : get_pdok(name) for name in filtered_names }

In [ ]:
[df.crs.to_epsg() for df in cbs.values() ]

In [ ]:
color_dict = {
    name: mcolors.to_hex(colors[i]) 
    for i, name in enumerate(sorted(set(gdf.statnaam)))
}

In [ ]:
import geopandas as gpd

# Load your GeoDataFrames (replace with your actual file paths)
gdf_provinces = cbs['gebiedsindelingen:provincie_gegeneraliseerd']
gdf_cities = cbs['gebiedsindelingen:gemeente_gegeneraliseerd']

# Ensure both GeoDataFrames use the same CRS (coordinate reference system)
gdf_cities = gdf_cities.to_crs(gdf_provinces.crs)

# Perform a spatial join to assign each city to a province
gdf_cities_partitioned = gpd.sjoin(gdf_cities, gdf_provinces, how="left", predicate="within")

In [ ]:
gdf_provinces

In [ ]:
gdf_cities_partitioned

In [ ]:
gdf_cities_partitioned.columns

In [ ]:
# Convert to dictionary: {province_name: [list of city names]}
province_city_dict = gdf_cities_partitioned.groupby("statnaam_right")["statnaam_left"].apply(list).to_dict()

In [ ]:
from itertools import chain

set(chain(*province_city_dict.values()))-set(gdf_cities.statnaam)


In [ ]:
gdf_cities_partitioned["color"] = gdf_cities_partitioned["statnaam_left"].map(color_dict)

map_all_cities_cbs = gdf_cities_partitioned.rename(
    columns={'statnaam_right':'Province','statnaam_left':'Municipality'}
).explore(
    color=gdf_cities_partitioned["color"],
    legend=False,
    tiles="OpenStreetMap",
    tooltip=['Municipality','Province'],
    zoom_on_bounds=True
)

In [ ]:
map_all_cities_cbs

In [ ]:
gdf_cities_partitioned["color"] = gdf_cities_partitioned["statnaam"].map(color_dict)

gdf.explore(
    color=gdf["color"],
    legend=False,
    tiles="OpenStreetMap",
    tooltip=['statnaam']
)

In [ ]:
m = gdf[gdf.statnaam.str.contains('Capelle')].explore(
    tooltip=False,
    style_kwds={'color': 'green', 'weight': 5, 'fill': False}
)

In [ ]:
folium_to_png(m,'test')

In [ ]:
if at_colab:
    from google.colab import files
    files.download("test.png")  # Replace with your file path